In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [6]:
# Load data
df = pd.read_csv("Reviews.csv.xz", compression="xz")
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [9]:
"""
Preprocess Data
    - Keep reviews with at least 5 total helpfulness votes
    - Remove malformed rows (nan rows)
    - Calculate helpfulness ratio
    - Binary labeling (majority finds it helpful)
"""
def preprocess_data(df: pd.DataFrame):
    df = df[df["HelpfulnessDenominator"] >= 5].copy()
    df = df.dropna(subset=["Text", "Score", "HelpfulnessNumerator", "HelpfulnessDenominator"])

    df["HelpfulnessRatio"] = (df["HelpfulnessNumerator"] / df["HelpfulnessDenominator"])
    df["Helpful"] = (df["HelpfulnessNumerator"] > df["HelpfulnessDenominator"] / 2).astype(int)

    print(f"Filtered dataset size: {len(df)}")
    print("\nClass distribution:")
    print(df["Helpful"].value_counts(normalize=True))

    return df

df = preprocess_data(df)
df.head()


Filtered dataset size: 67467

Class distribution:
Helpful
1    0.793662
0    0.206338
Name: proportion, dtype: float64


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,HelpfulnessRatio,Helpful
14,15,B001GVISJM,A2MUGFV2TDQ47K,"Lynrie ""Oh HELL no""",4,5,5,1268352000,Strawberry Twizzlers - Yummy,The Strawberry Twizzlers are my guilty pleasur...,0.8,1
15,16,B001GVISJM,A1CZX3CP8IKQIJ,Brian A. Lee,4,5,5,1262044800,"Lots of twizzlers, just what you expect.",My daughter loves twizzlers and this shipment ...,0.8,1
32,33,B001EO5QW8,AOVROBZ8BNTP7,S. Potter,19,19,4,1163376000,Best of the Instant Oatmeals,McCann's Instant Oatmeal is great if you must ...,1.0,1
33,34,B001EO5QW8,A3PMM0NFVEJGK9,"Megan ""Bad at Nicknames""",13,13,4,1166313600,Good Instant,This is a good instant oatmeal from the best o...,1.0,1
34,35,B001EO5QW8,A2EB6OGOWCRU5H,CorbyJames,9,9,5,1175212800,Great Irish oatmeal for those in a hurry!,Instant oatmeal can become soggy the minute th...,1.0,1


In [10]:
"""
Create the TF-IDF and rating features
    - Transform review text into TF-IDF features
    - Encode star ratings as five one-hot features
"""
preprocessor = ColumnTransformer(
    transformers=[
        (
            "text", # Name of tramsformer
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                max_features=20000, # Helps control memory usage
                min_df=5,
                max_df=0.90,
                ngram_range=(1, 1), # Could maybe try (1, 2) to capture bigrams
                sublinear_tf=True # Dampens influence of repeated words
            ),
            "Text" # Column being transformed from df
        ),
        (
            "rating",
            OneHotEncoder(
                categories=[[1, 2, 3, 4, 5]],
                handle_unknown="ignore"
            ),
            ["Score"]
        )
    ]
)

In [14]:
def split_data(X: pd.DataFrame, y: pd.DataFrame, test_size: float):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)

    print(f"Training reviews: {len(X_train)}")
    print(f"Testing reviews: {len(X_test)}")

    return X_train, X_test, y_train, y_test

In [16]:
# Inputs and Targets
X = df[["Text", "Score"]]
y = df["Helpful"]

X_train, X_test, y_train, y_test = split_data(X, y, 0.20)

# Train the model
model = Pipeline([
    ("features", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        penalty="l2",
        class_weight="balanced",
        random_state=42
    ))
])
model.fit(X_train, y_train)

# Predict labels on the test set
y_pred = model.predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not helpful", "Helpful"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training reviews: 53973
Testing reviews: 13494


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy:  0.8134726545131169
F1 Score:  0.873026282601019

Classification report:
              precision    recall  f1-score   support

 Not helpful       0.53      0.83      0.65      2784
     Helpful       0.95      0.81      0.87     10710

    accuracy                           0.81     13494
   macro avg       0.74      0.82      0.76     13494
weighted avg       0.86      0.81      0.83     13494

Confusion Matrix:
[[2324  460]
 [2057 8653]]


In [19]:
# Analyze words and their relationship to helpfulness

# Feature names
feature_names = model.named_steps["features"].get_feature_names_out()

# Learned logistic regression coefficients
coefficients = model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

# Most positive and negative features
print("\nTop 10 features associated with helpfulness:")
print(coef_df.sort_values("coefficient", ascending=False).head(10).to_string(index=False))

print("\nTop 10 features associated with not helpful:")
print(coef_df.sort_values("coefficient", ascending=True).head(10).to_string(index=False))

# Isolate the five star-rating features
rating_coef = coef_df[coef_df["feature"].str.startswith("rating_")].copy()
print("\nRating coefficients:")
print(rating_coef.sort_values("feature").to_string(index=False))


Top 10 features associated with helpfulness:
            feature  coefficient
        text__china     3.446148
      text__changed     3.429744
        text__moldy     3.035697
          text__toy     3.003497
        text__blend     2.796380
   text__acceptable     2.712065
      text__evening     2.673149
  text__concentrate     2.662890
text__disappointing     2.657438
        text__plant     2.588485

Top 10 features associated with not helpful:
        feature  coefficient
      text__blk    -2.903391
      text__tan    -2.802832
       text__im    -2.642052
     text__yoga    -2.566255
 text__triangle    -2.530610
  text__vermont    -2.512261
    text__early    -2.511143
    text__kitty    -2.486480
text__following    -2.457861
text__neighbors    -2.375795

Rating coefficients:
        feature  coefficient
rating__Score_1    -1.341821
rating__Score_2    -1.205387
rating__Score_3    -0.677705
rating__Score_4     0.792110
rating__Score_5     1.550855
